<a href="https://colab.research.google.com/github/RaquelHernanz/BachelorsThesis_SyntheticClinicalData/blob/master/NOTEBOOK5_BTSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 5 — Pre-Generation Data Transformations**

- **Author:** Raquel Hernanz Hernández
- **Supervisors:** José María Herrera and Guillermo José Ortega
- **Degree:** Biomedical Engineering  
- **Project:** Bachelor Thesis — *Generation and Validation of Synthetic Data from a Hospital Emergency Department*  

---
## Role in the pipeline

| Notebook | Purpose |
|----------|---------|
| **NB5 (this notebook)** | Pre-generation data transformations |
| NB6 | Synthetic data generation — CTAB-GAN+ |
| NB7 | Synthetic data generation — ARF/FORDE |

---

## Objectives

Transform the cleaned dataset (`dataset_FINAL.csv`, produced by **NB1** and analyzed in **NB2**–**NB4**) into a format ready for synthetic data generation (**CTAB-GAN+** or **ARF**). The three mortality endpoints (`Mort. 2D`, `Mort. 7D`, `Mort. 30D`) exhibit low prevalence (4.80%–10.98%), so every transformation is designed to preserve class-conditional structure and tail fidelity.

**Key transformations in this notebook:**
- Integrity verification and variable classification
- Log1p transformation for right-skewed continuous variables
- Min-Max normalisation for continuous features
- Metadata export (variable types, scaler parameters)
- Preprocessed CSV export for downstream generation notebooks

## **1. Libraries**

| Library | Role |
|---------|------|
| `pandas`, `numpy` | Data handling and numerical operations |
| `sklearn.preprocessing` | `MinMaxScaler` for [0, 1] normalisation |
| `openpyxl` | Read the `.xlsx` attribute table from Drive |
| `matplotlib`, `seaborn` | Distribution comparison plots (before vs. after) |
| `json` | Export transformation metadata as a portable JSON file |
| `warnings` | Suppress non-critical runtime warnings |

In [ ]:
"""
PURPOSE : Import all libraries required by NB5.

Libraries
---------
pandas, numpy          — data handling and numerical operations.
sklearn.preprocessing  — MinMaxScaler for [0, 1] feature normalisation.
matplotlib, seaborn    — distribution comparison plots (§§8.1, 12.1).
json                   — export transformation metadata as a portable JSON file.
warnings               — suppress non-critical FutureWarning messages.
"""

from __future__ import annotations
import numpy as np
import pandas as pd
import json
import warnings
from typing import Dict, List, Tuple

from sklearn.preprocessing import MinMaxScaler

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

print("Libraries loaded successfully.")

Libraries loaded successfully.


### Global plot style

All figures in this notebook use the **TFG unified style**, a `theme_bw()`-equivalent
for matplotlib / seaborn, consistent across NB2, NB3, NB4, and NB5.

**Key visual conventions:**
- White background, light grey grid, thin black panel border.
- Serif font family (DejaVu Serif / Times New Roman), 11 pt base size.
- `TFG_COLORS` palette:
  - `"before"` (#3366CC) — original distributions.
  - `"after"` (#d62728) — transformed distributions.
  - `"highlight"` (#2ca02c) — reference markers.
- Saved figures: 150 dpi, tight bounding box.


In [ ]:
# ── TFG Unified Plot Style (Zhang et al. 2017 theme_bw equivalent) ────────────
TFG_STYLE = {
    "axes.facecolor":    "white",
    "figure.facecolor":  "white",
    "axes.edgecolor":    "black",
    "axes.linewidth":    0.8,
    "axes.grid":         True,
    "grid.color":        "#d9d9d9",
    "grid.linewidth":    0.5,
    "grid.linestyle":    "-",
    "font.family":       "serif",
    "font.serif":        ["DejaVu Serif", "Times New Roman", "serif"],
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   9,
    "legend.frameon":    True,
    "legend.framealpha": 0.9,
    "legend.edgecolor":  "#cccccc",
    "xtick.direction":   "out",
    "ytick.direction":   "out",
    "xtick.major.size":  4,
    "ytick.major.size":  4,
    "figure.dpi":        100,
    "savefig.dpi":       150,
    "savefig.bbox":      "tight",
}
mpl.rcParams.update(TFG_STYLE)
sns.set_theme(style="white", rc=TFG_STYLE)

TFG_COLORS = {
    "survivors":     "#FFD700",
    "non_survivors": "#87CEEB",
    "before":        "#3366CC",
    "after":         "#d62728",
    "highlight":     "#2ca02c",
}

print("TFG plot style applied.")

TFG plot style applied.


## **2. Data loading**

Mount at local or Google Drive and load:
- `dataset_FINAL.csv` — the cleaned dataset produced by **NB1** (2 376 × 33).
- `TABLA_ATRIBUTOS_ACTUALIZADO.xlsx` — the prehospital attribute dictionary.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── Paths ─────────────────────────────────────────────────────────────
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/TFG/dataset_FINAL.csv"
ATTR_PATH = "/content/drive/MyDrive/Colab Notebooks/TFG/TABLA_ATRIBUTOS_ACTUALIZADO.xlsx"

# ── Load dataset ──────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

Mounted at /content/drive
Dataset loaded: 2376 rows × 33 columns


## **3. Transformation configuration**

Central switch panel for **NB5**. Set each boolean to `True` to apply the
corresponding transformation, or `False` to skip it. The downstream cells
read these flags and execute conditionally, no other edits are needed.

| Flag | Controls |
|------|----------|
| `CFG_DROP_TAM` | Drop `TAM` (multicollinearity with TAS/TAD, **Section 6**) |
| `CFG_LOG1P` | `log1p` skewness reduction on Lactato & Glucemia (**Section 8**) |
| `CFG_INT_CAST` | Integer-type enforcement on GCS ordinal columns (**Section 9**) |
| `CFG_OHE` | One-Hot Encoding of `Ritmo` and `ST` (**Section 10**) |
| `CFG_MINMAX` | Min-Max normalisation of continuous features (**Section 11**) |
| `CFG_PLOT` | Distribution comparison plots (**Sections 8.1 and 12.1**) |

> **Dependency note:** `CFG_MINMAX` applies after `CFG_LOG1P`; disabling
> `CFG_LOG1P` while keeping `CFG_MINMAX` active is valid but will normalise
> the raw (unskewed) values of Lactato and Glucemia.  
> Disabling `CFG_OHE` keeps `Ritmo` and `ST` as raw integer codes,  ensure
> the downstream generator supports integer categoricals.

In [ ]:
"""
PURPOSE : Central configuration panel for NB5 transformations.
          Set each CFG_* boolean to True/False to enable/disable each step.
          All downstream cells read these flags; no other edits are required.
"""

# ══════════════════════════════════════════════════════════════════════════
# NB5 — TRANSFORMATION CONFIGURATION
# Edit the booleans below. All downstream cells read these flags.
# ══════════════════════════════════════════════════════════════════════════

# ── Multicollinearity resolution ────────────────────────────────────────────
# Drop TAM (mean arterial pressure): arithmetic combination of TAS and TAD.
CFG_DROP_TAM: bool = False

# ── Skewness reduction ──────────────────────────────────────────────────────
# Apply log1p to Lactato and Glucemia (right-skewed, skew > 2.5).
CFG_LOG1P: bool = False

# ── Ordinal type enforcement ────────────────────────────────────────────────
# Cast GCS columns to int64 (CTAB-GAN+ treats them as discrete/ordinal, not continuous).
CFG_INT_CAST: bool = True

# ── Nominal encoding ────────────────────────────────────────────────────────
# One-hot encode Ritmo (≤21 classes) and ST (7 classes).
CFG_OHE: bool = True

# ── Feature scaling ─────────────────────────────────────────────────────────
# Min-Max normalisation of all continuous columns to [0, 1].
CFG_MINMAX: bool = False

# ── Visualisation ───────────────────────────────────────────────────────────
# Generate distribution comparison plots (7.1 and 11.1).
CFG_PLOT: bool = False

# ── Thesis run configuration (NB6b CTAB-GAN+ / NB7 ARF) ─────────────
# The following configuration was used to produce the dataset_PREGEN.csv
# consumed by NB6b and NB7:
#
#   CFG_DROP_TAM = False   TAM retained; removed by VIF in NB4 scoring only
#   CFG_LOG1P    = False   Skewness acceptable for CTAB-GAN+ MSN internals
#   CFG_INT_CAST = True    GCS columns as integers
#   CFG_OHE      = True    Ritmo/ST one-hot encoded
#   CFG_MINMAX   = False   Does not need min-max normalization
#
# To reproduce: set the flags above to match and re-run all cells.

# ── Configuration summary ───────────────────────────────────────────────────
_CFG_MAP = {
    "Drop TAM (multicollinearity)":        CFG_DROP_TAM,
    "Log1p (Lactato / Glucemia)": CFG_LOG1P,
    "Integer cast (GCS ordinals)":     CFG_INT_CAST,
    "One-Hot Encoding (Ritmo / ST)":   CFG_OHE,
    "MinMax normalisation [0,1]":       CFG_MINMAX,
    "Distribution plots":              CFG_PLOT,
}

_W = max(len(k) for k in _CFG_MAP) + 2
print("\n" + "─" * (_W + 18))
print(f"  NB5 — Active transformation configuration")
print("─" * (_W + 18))
for name, flag in _CFG_MAP.items():
    mark   = "✓" if flag else "✗"
    status = "ENABLED " if flag else "DISABLED"
    print(f"  {mark}  {name:<{_W}} {status}")
print("─" * (_W + 18) + "\n")



─────────────────────────────────────────────────
  NB5 — Active transformation configuration
─────────────────────────────────────────────────
  ✗  Drop TAM (multicollinearity)    DISABLED
  ✗  Log1p (Lactato / Glucemia)      DISABLED
  ✓  Integer cast (GCS ordinals)     ENABLED 
  ✓  One-Hot Encoding (Ritmo / ST)   ENABLED 
  ✗  MinMax normalisation [0,1]      DISABLED
  ✗  Distribution plots              DISABLED
─────────────────────────────────────────────────



## **4. Integrity verification**

Cross-notebook check following the same protocol as **NB2**. We verify:

1. **Shape** matches NB1 output (2 376 × 33).
2. **No residual NaN** after NB1 cleaning.
3. **Mortality monotonicity** constraint: `Mort. 2D ≤ Mort. 7D ≤ Mort. 30D` per row.
4. **Data types** are consistent (no object columns that should be numeric).

> Running integrity checks catches data corruption during export-import cycles (encoding errors, column truncation) before any transformation is applied.

In [ ]:
"""
PURPOSE : Cross-notebook integrity verification (mirrors NB2 protocol).

Checks performed
----------------
1. Row count matches NB1 output (2 376 rows).
2. No residual NaN values.
3. Mortality monotonicity: Mort. 2D <= Mort. 7D <= Mort. 30D per row.
4. Data-type summary for visual inspection.
"""

# ── 3.1 Shape check ───────────────────────────────────────────────────
EXPECTED_N_ROWS = 2376  # defined in config block
if df.shape[0] != EXPECTED_N_ROWS:
    print(f"⚠ Row count {df.shape[0]} differs from expected {EXPECTED_N_ROWS}.")
else:
    print(f"✓ Rows: {df.shape[0]}")

# ── 3.2 NaN check ────────────────────────────────────────────────────
nan_total = df.isna().sum().sum()
assert nan_total == 0, f"Found {nan_total} NaN values"
print(f"✓ NaN count: {nan_total}")

# ── 3.3 Mortality monotonicity ───────────────────────────────────────
mort_cols = ["Mort. 2D", "Mort. 7D", "Mort. 30D"]
violations = df.loc[
    (df["Mort. 2D"] > df["Mort. 7D"]) | (df["Mort. 7D"] > df["Mort. 30D"])
]
assert len(violations) == 0, f"Found {len(violations)} monotonicity violations"
print(f"✓ Mortality monotonicity: 0 violations")

# ── 3.4 Dtype summary ────────────────────────────────────────────────
print(f"\nData types:\n{df.dtypes.value_counts().to_string()}")
print(f"\nColumn list:\n{list(df.columns)}")

✓ Rows: 2376
✓ NaN count: 0
✓ Mortality monotonicity: 0 violations

Data types:
int64      29
float64     4

Column list:
['Edad', 'Sexo', 'FR', 'SpO2', 'O2', 'TAS', 'TAD', 'TAM', 'FC', 'TT', 'GCS.O', 'GCS.V', 'GCS.M', 'GCS', 'Lactato', 'Glucemia', 'Ritmo', 'ST', 'O2 sup.', 'FiO2', 'Gafas', 'Venturi', 'Resrv.', 'Nebul.', 'VNI', 'IOT', 'VAD', 'VM', 'MAVA', 'TTE', 'Mort. 2D', 'Mort. 7D', 'Mort. 30D']


## **5. Variable classification**

Before applying any transformation, we classify every column into one of three types:

- **Continuous:** real-valued physiological measurements (e.g., *TAS*, *Lactato*, *Edad*).
- **Binary:** dichotomous flags already encoded as {0, 1} in **NB1** (e.g., *Sexo*, *O2*, *Mort. 2D*).
- **Ordinal / bounded integer:** discrete variables with a natural order and bounded range (e.g., *GCS*, *GCS.V*, *GCS.M*, *GCS.O*).

> CTAB-GAN+ uses internal encoders for continuous and discrete variables. Continuous columns pass through **Mode-Specific Normalisation** (MSN), while discrete columns use conditional **one-hot encoding**.

In [ ]:
"""
PURPOSE : Classify every column as continuous, binary, ordinal, or categorical.

Input   : df — the loaded dataset_FINAL.csv.
Output  : CONTINUOUS_COLS, BINARY_COLS, ORDINAL_COLS, CATEGORICAL_COLS,
          ALL_MORT_COLS — used by all downstream transformation cells.
Method  : Lists defined by domain knowledge (TABLA_ATRIBUTOS_ACTUALIZADO.xlsx).
          A set-difference assertion verifies that every column is classified.
"""

# ── Variable classification based on domain knowledge ─────────────────
# These lists are derived from the attribute table (TABLA_ATRIBUTOS_ACTUALIZADO.xlsx).

ALL_MORT_COLS = ["Mort. 2D", "Mort. 7D", "Mort. 30D"]

BINARY_COLS = [
    # Demographic
    "Sexo",
    # Oxygen therapy
    "O2", "Gafas", "Resrv.", "Venturi", "O2 sup.",
    # Ventilatory support
    "MAVA", "VM", "IOT", "VNI",
    # Other interventions
    "VAD", "Nebul.", "TTE",
]

# Nominal categorical (multi-class) — will be one-hot encoded in Section 9
CATEGORICAL_COLS = ["Ritmo", "ST"]

ORDINAL_COLS = [
    "GCS", "GCS.V", "GCS.M", "GCS.O",
]

# Everything else (excluding outcomes, binary, categorical, ordinal) is continuous.
# Includes: Edad, TAS, TAD, FC, FR, TT, SpO2, FiO2, Lactato, Glucemia
CONTINUOUS_COLS = sorted([
    c for c in df.columns
    if c not in ALL_MORT_COLS + BINARY_COLS + CATEGORICAL_COLS + ORDINAL_COLS
])

print(f"Continuous   ({len(CONTINUOUS_COLS)}): {CONTINUOUS_COLS}")
print(f"Binary       ({len(BINARY_COLS)}): {BINARY_COLS}")
print(f"Categorical  ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")
print(f"Ordinal      ({len(ORDINAL_COLS)}): {ORDINAL_COLS}")
print(f"Outcomes     ({len(ALL_MORT_COLS)}): {ALL_MORT_COLS}")
print(f"Total:       {len(CONTINUOUS_COLS)+len(BINARY_COLS)+len(CATEGORICAL_COLS)+len(ORDINAL_COLS)+len(ALL_MORT_COLS)} / {df.shape[1]} columns")

# Sanity: all columns accounted for
all_classified = set(CONTINUOUS_COLS + BINARY_COLS + CATEGORICAL_COLS + ORDINAL_COLS + ALL_MORT_COLS)
assert all_classified == set(df.columns), f"Unclassified: {set(df.columns) - all_classified}"
print("\n✓ All columns classified.")

Continuous   (11): ['Edad', 'FC', 'FR', 'FiO2', 'Glucemia', 'Lactato', 'SpO2', 'TAD', 'TAM', 'TAS', 'TT']
Binary       (13): ['Sexo', 'O2', 'Gafas', 'Resrv.', 'Venturi', 'O2 sup.', 'MAVA', 'VM', 'IOT', 'VNI', 'VAD', 'Nebul.', 'TTE']
Categorical  (2): ['Ritmo', 'ST']
Ordinal      (4): ['GCS', 'GCS.V', 'GCS.M', 'GCS.O']
Outcomes     (3): ['Mort. 2D', 'Mort. 7D', 'Mort. 30D']
Total:       33 / 33 columns

✓ All columns classified.


## **6. Pre-transformation descriptive summary**

Snapshot of the dataset before any transformation, for comparison with the post-transformation state in **Section 10**.

In [ ]:
# ── Continuous + ordinal descriptive statistics ───────────────────────
desc_before = df[CONTINUOUS_COLS + ORDINAL_COLS].describe().T
desc_before["skew"] = df[CONTINUOUS_COLS + ORDINAL_COLS].skew()
desc_before["kurtosis"] = df[CONTINUOUS_COLS + ORDINAL_COLS].kurtosis()

print("Pre-transformation statistics (continuous + ordinal):")
display(desc_before.round(3))

Pre-transformation statistics (continuous + ordinal):


,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
Edad,2376.0,66.114,18.114,18.000,54.000,69.00,81.000,98.000,-0.609,-0.352
FC,2376.0,89.739,29.998,16.000,70.000,85.00,103.000,267.000,1.194,2.698
FR,2376.0,19.628,8.426,3.000,14.000,18.00,23.000,60.000,1.304,1.545
FiO2,2376.0,0.304,0.205,0.210,0.210,0.21,0.280,1.000,2.643,6.071
Glucemia,2376.0,144.099,60.101,22.000,107.000,128.00,163.000,640.000,2.577,10.519
Lactato,2376.0,3.549,2.575,0.900,2.000,2.90,4.100,20.400,2.644,8.958
SpO2,2376.0,93.327,8.604,45.000,93.000,96.00,98.000,100.000,-2.925,9.706
TAD,2376.0,79.307,18.337,20.000,68.000,80.00,90.000,183.000,0.112,0.678
TAM,2376.0,98.854,20.963,27.333,86.333,99.00,111.667,195.333,0.098,0.651
TAS,2376.0,137.982,30.640,40.000,119.000,137.00,155.250,261.000,0.282,0.387


## **7. Multicollinearity resolution**

**NB2** (Spearman correlation) and **NB4** (VIF analysis) identified near-arithmetic collinearity among the three blood pressure variables:

| Pair | Spearman ρ |
|------|-----------|
| TAD – TAM | 0.932 |
| TAS – TAM | 0.904 |
| TAS – TAD | 0.703 |

**TAM** (mean arterial pressure) is a linear combination of **TAS** and **TAD** by definition: `TAM ≈ (TAS + 2·TAD) / 3`. Retaining all three would cause:
- **For CTAB-GAN+:** the generator might learn to produce synthetic rows where TAM is inconsistent with TAS and TAD, creating clinically implausible records.
- **For ARF:** if TAM is placed after TAS and TAD in the generation order, the RF model would effectively learn the arithmetic formula rather than clinical signal, wasting model capacity.

> Drop `TAM`, retaining `TAS` and `TAD` as the independent blood pressure measurements.

In [ ]:
# ── §7 — Drop TAM (multicollinearity) ───────────────────────────────────
# Controlled by CFG_DROP_TAM
_dropped_cols: dict = {}  # populated here; consumed in §12 metadata

COL_TO_DROP = "TAM"

if CFG_DROP_TAM:
    if COL_TO_DROP in df.columns:
        df = df.drop(columns=[COL_TO_DROP])
        CONTINUOUS_COLS = [c for c in CONTINUOUS_COLS if c != COL_TO_DROP]
        _dropped_cols[COL_TO_DROP] = (
            "Near-arithmetic collinearity with TAS/TAD (rho>0.90). "
            "TAM = (TAS + 2*TAD)/3 by definition."
        )
        print(f"✓ Dropped '{COL_TO_DROP}'. New shape: {df.shape}")
    else:
        print(f"⚠ '{COL_TO_DROP}' not found — skipping drop.")
else:
    print(f"— CFG_DROP_TAM=False: '{COL_TO_DROP}' retained.")

print(f"Continuous columns ({len(CONTINUOUS_COLS)}): {CONTINUOUS_COLS}")


— CFG_DROP_TAM=False: 'TAM' retained.
Continuous columns (11): ['Edad', 'FC', 'FR', 'FiO2', 'Glucemia', 'Lactato', 'SpO2', 'TAD', 'TAM', 'TAS', 'TT']


## **8. Log1p transformation for right-skewed variables**

**NB2** identified **Lactato** and **Glucemia** as markedly right-skewed continuous variables. The `log1p` transformation (`log(1 + x)`) can be applied to compress extreme values while preserving zero values without imputation.

> *Why log1p and not log?* ->
> Several patients have Lactato values near or at zero. `log(0)` is undefined, so `log(1 + x)` is the standard solution — it maps 0 → 0, preserves the order of values, and is perfectly invertible via `expm1(y)`. This is critical: after synthetic data generation, we must apply the inverse to recover clinically meaningful units (mmol/L for Lactato, mg/dL for Glucemia).

> *Do our models have mechanisms to deal with skewed variables?*
> - **CTAB-GAN+:** MSN fits Gaussian mixture modes.
> - **ARF:** Tree splits on the log-scale could produce more balanced partitions, improving conditional distribution accuracy in the tails, precisely where critical patients reside (**NB3** outlier analysis).

> *Is it neccesary this transformation?* → No, both generation models can work properly without prior Lop1p on *Glucemia* and *Lactato*.

In [ ]:
# ── §8 — Log1p transformation ────────────────────────────────────────────
# Controlled by CFG_LOG1P
LOG_COLS = ["Lactato", "Glucemia"]
log_metadata: dict = {}  # populated here; consumed in §12 metadata

if CFG_LOG1P:
    for col in LOG_COLS:
        if col in df.columns:
            original_skew = df[col].skew()
            df[col] = np.log1p(df[col])
            new_skew = df[col].skew()
            log_metadata[col] = {
                "transform": "log1p",
                "inverse": "expm1",
                "original_skew": round(float(original_skew), 4),
                "transformed_skew": round(float(new_skew), 4),
            }
            print(f"  {col}: skewness {original_skew:.3f} → {new_skew:.3f}")
    print(f"\n✓ Log1p applied to {len(log_metadata)} column(s).")
else:
    print("— CFG_LOG1P=False: log1p transformation skipped.")


— CFG_LOG1P=False: log1p transformation skipped.


### 8.1 Visual comparison: raw vs. log1p-transformed distributions

Side-by-side histograms confirm the skewness reduction. Left: original scale (loaded from a fresh copy); right: log1p-transformed (current `df`).

In [ ]:
# ── §8.1 — Log1p distribution plots ──────────────────────────────────────
# Controlled by CFG_LOG1P and CFG_PLOT
if CFG_LOG1P and CFG_PLOT and log_metadata:
    df_raw = pd.read_csv(DATA_PATH)

    fig, axes = plt.subplots(len(log_metadata), 2,
                              figsize=(10, 4 * len(log_metadata)))
    if len(log_metadata) == 1:
        axes = axes.reshape(1, -1)

    for i, col in enumerate(log_metadata):
        # Original distribution
        axes[i, 0].hist(df_raw[col], bins=50, color=TFG_COLORS["before"],
                         edgecolor="white", alpha=0.85)
        axes[i, 0].set_title(f"{col} — original (skew={df_raw[col].skew():.2f})")
        axes[i, 0].set_xlabel(col)
        axes[i, 0].set_ylabel("Frequency")

        # Log1p-transformed distribution
        axes[i, 1].hist(df[col], bins=50, color=TFG_COLORS["after"],
                         edgecolor="white", alpha=0.85)
        axes[i, 1].set_title(f"{col} — log1p (skew={df[col].skew():.2f})")
        axes[i, 1].set_xlabel(f"log(1 + {col})")
        axes[i, 1].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()
    del df_raw
elif not CFG_LOG1P:
    print("— CFG_LOG1P=False: log1p plots skipped.")
elif not CFG_PLOT:
    print("— CFG_PLOT=False: log1p plots skipped.")


— CFG_LOG1P=False: log1p plots skipped.


## **9. Ordinal variable treatment (GCS components)**

*GCS* and its sub-scores (*GCS.V*, *GCS.M*, *GCS.O*) exhibit a pronounced ceiling effect: median = Q3 = 15 for GCS total. These are bounded integers, not continuous measurements.

| Variable | Range | Unique values |
|----------|-------|---------------|
| GCS      | 3–15  | 13 |
| GCS.V    | 1–5   | 5  |
| GCS.M    | 1–6   | 6  |
| GCS.O    | 1–4   | 4  |

**Treatment decision:** Keep GCS variables as **integer type** and flag them as **discrete/ordinal** in the metadata.

> - **CTAB-GAN+:** Discrete columns are processed through a conditional encoder rather than MSN. This prevents the generator from producing fractional GCS values (e.g., *GCS* = 12.7), which are clinically impossible.
> - **ARF:** Each GCS column will be modelled as a classification target (predicting one of k integer classes), naturally respecting the bounded domain.

> *Is it neccesary this transformation?* → This transformation is active, although not require, as the GCS only contain integer values from [3, 15].

In [ ]:
# ── §9 — Ordinal integer cast (GCS components) ───────────────────────────
# Controlled by CFG_INT_CAST
if CFG_INT_CAST:
    for col in ORDINAL_COLS:
        if col in df.columns:
            df[col] = df[col].astype(int)
            n_unique = df[col].nunique()
            print(f"  {col}: dtype={df[col].dtype}, "
                  f"unique={n_unique}, range=[{df[col].min()}, {df[col].max()}]")
    print(f"\n✓ Ordinal columns cast to integer type.")
else:
    print("— CFG_INT_CAST=False: ordinal integer cast skipped.")


  GCS: dtype=int64, unique=13, range=[3, 15]
  GCS.V: dtype=int64, unique=5, range=[1, 5]
  GCS.M: dtype=int64, unique=6, range=[1, 6]
  GCS.O: dtype=int64, unique=4, range=[1, 4]

✓ Ordinal columns cast to integer type.


## **10. One-Hot Encoding of nominal categorical variables**

`Ritmo` (cardiac rhythm) and `ST` (ST-segment pattern) are nominal categorical variables with **21** and **7** classes respectively, they have no natural ordinal relationship, so they cannot be passed as integers to either generative model.

| Variable | Type | Classes | Range |
|----------|------|---------|-------|
| `Ritmo` | Nominal | 21 | 1 Sinusal … 21 Otros |
| `ST` | Nominal | 7 | 1 NO … 7 Otros |

**Treatment:** Full one-hot encoding (all *k* dummy columns retained, `drop_first=False`) to preserve every category for the generator.

> *Why not drop the first dummy?* In ordinary regression, dropping one category avoids perfect multicollinearity. In synthetic data generation, the generator must reproduce every category; dropping a column would make category 1 unrecoverable from the output.

> *Reversibility:* The inverse transform is `argmax` over the *k* OHE columns → integer code → original label. Label mappings are exported in the metadata JSON.

> *Is it neccesary this transformation?* → Yes, at least for CTAB-GAN+. The transformation help to yield better results in the fidelity of both features in the fidelity test.

In [ ]:
# ── §10 — One-Hot Encoding of nominal categoricals ────────────────────────
# Controlled by CFG_OHE

# Label dictionaries (integer code → readable name)
RITMO_LABELS = {
    1: "Sinusal",          2: "Arritmia_sinusal",  3: "FA",
    4: "Flutter",          5: "Taquicardia",       6: "TSV",
    7: "TV",               8: "Bradicardia",       9: "BloqAV1",
    10: "BloqAV2I",        11: "BloqAV2II",        12: "BloqAV_completo",
    13: "Marcapasos",      14: "Union",            15: "Idioventricular",
    16: "BRD",             17: "BRI",              18: "Extrasistoles",
    19: "Asistolia",       20: "FV",               21: "Otros",
}

ST_LABELS = {
    1: "No",       2: "Elevacion",    3: "Descenso",
    4: "T_picudas", 5: "T_negativas", 6: "OndaQ",
    7: "Otros",
}

OHE_DEFS = {"Ritmo": RITMO_LABELS, "ST": ST_LABELS}

# Initialise tracking lists (empty if CFG_OHE=False)
ohe_metadata: dict = {}
ritmo_ohe_cols: list = []
st_ohe_cols:   list = []

if CFG_OHE:
    for orig_col, labels in OHE_DEFS.items():
        # Use str keys to survive JSON round-trip (json.dump converts int keys to str)
        observed = [int(v) for v in sorted(df[orig_col].dropna().unique())]
        col_map: dict = {}
        for val in observed:
            label    = labels.get(val, str(val))
            col_name = f"{orig_col}_{label}"
            df[col_name] = (df[orig_col] == val).astype(int)
            col_map[str(val)] = col_name
            if orig_col == "Ritmo":
                ritmo_ohe_cols.append(col_name)
            else:
                st_ohe_cols.append(col_name)
        ohe_metadata[orig_col] = {
            "original_dtype": "int (categorical)",
            "n_classes":      len(observed),
            "observed_codes": observed,
            "column_map":     col_map,
            "inverse": "argmax over OHE columns → int(key) → original code",
        }
        df = df.drop(columns=[orig_col])
        _dropped_cols[orig_col] = f"Replaced by OHE ({len(observed)} classes)."
        print(f"  {orig_col}: {len(observed)} classes → {len(col_map)} OHE columns")

    print(f"Ritmo OHE cols ({len(ritmo_ohe_cols)}): {ritmo_ohe_cols}")
    print(f"ST    OHE cols ({len(st_ohe_cols)}):    {st_ohe_cols}")
    print(f"New shape after OHE: {df.shape}")
    print("✓ One-Hot Encoding complete.")
else:
    print("— CFG_OHE=False: Ritmo and ST kept as integer codes.")


  Ritmo: 18 classes → 18 OHE columns
  ST: 6 classes → 6 OHE columns
Ritmo OHE cols (18): ['Ritmo_Sinusal', 'Ritmo_Arritmia_sinusal', 'Ritmo_FA', 'Ritmo_Flutter', 'Ritmo_Taquicardia', 'Ritmo_TSV', 'Ritmo_TV', 'Ritmo_Bradicardia', 'Ritmo_BloqAV1', 'Ritmo_BloqAV2I', 'Ritmo_BloqAV2II', 'Ritmo_BloqAV_completo', 'Ritmo_Marcapasos', 'Ritmo_Union', 'Ritmo_Idioventricular', 'Ritmo_BRD', 'Ritmo_BRI', 'Ritmo_Extrasistoles']
ST    OHE cols (6):    ['ST_No', 'ST_Elevacion', 'ST_Descenso', 'ST_T_picudas', 'ST_T_negativas', 'ST_OndaQ']
New shape after OHE: (2376, 55)
✓ One-Hot Encoding complete.


## **11. Min-Max normalisation of continuous variables**

Continuous features are scaled to **[0, 1]** using `MinMaxScaler`. In general, this is a pre-processing step recommended for GAN-based generators (Hernández et al., 2022).

> *Why Min-Max and not Standard Scaling?* →
> Starting from a [0, 1] range ensures all features are in a comparable scale, improving mode estimation stability

> *Do our models implement any normalization process?*
> - CTAB-GAN+ internally applies MSN *on top of* the input.
> - For ARF, tree-based models are scale-invariant, normalisation does not affect splits.
>
> *Reversibility:* The scaler's `min_` and `scale_` parameters are exported in Section 11 so that synthetic data can be inverse-transformed to the original clinical units after generation.

> *Is it neccesary this transformation?* → No, both generation models can work properly without prior normalization.

In [ ]:
# ── §11 — Min-Max normalisation [0, 1] ───────────────────────────────────
# Controlled by CFG_MINMAX
scaler_metadata: dict = {}  # populated here; consumed in §12 metadata

if CFG_MINMAX:
    scaler = MinMaxScaler(feature_range=(0, 1))
    df[CONTINUOUS_COLS] = scaler.fit_transform(df[CONTINUOUS_COLS])

    scaler_metadata = {
        col: {
            "min":        round(float(scaler.data_min_[i]),   6),
            "max":        round(float(scaler.data_max_[i]),   6),
            "scale":      round(float(scaler.scale_[i]),      6),
            "data_range": round(float(scaler.data_range_[i]), 6),
        }
        for i, col in enumerate(CONTINUOUS_COLS)
    }

    print("Post-normalisation ranges:")
    for col in CONTINUOUS_COLS:
        print(f"  {col}: [{df[col].min():.4f}, {df[col].max():.4f}]")
    print(f"\n✓ MinMaxScaler applied to {len(CONTINUOUS_COLS)} continuous columns.")
else:
    print("— CFG_MINMAX=False: Min-Max normalisation skipped.")


— CFG_MINMAX=False: Min-Max normalisation skipped.


## **12. Post-transformation summary**

Compare the dataset before and after transformations. Key checks:
- Continuous columns now in [0, 1].
- Binary columns unchanged ({0, 1}).
- Ordinal columns unchanged (integer range preserved).
- Outcome columns unchanged.
- Mortality event rates unchanged (transformations must not alter the label distribution).

In [ ]:
# ── Descriptive stats after transformations ────────────────────────────
desc_after = df[CONTINUOUS_COLS + ORDINAL_COLS].describe().T
desc_after["skew"] = df[CONTINUOUS_COLS + ORDINAL_COLS].skew()

print("Post-transformation statistics (continuous + ordinal):")
print(desc_after.round(3).to_string())

# ── Mortality rates (must be identical to NB1 output) ─────────────────
print("\nMortality event rates (must match NB1):")
for col in ALL_MORT_COLS:
    rate = df[col].mean() * 100
    n_events = int(df[col].sum())
    print(f"  {col}: {n_events} events ({rate:.2f}%)")

# ── Binary column check ──────────────────────────────────────────────
print("\nBinary columns — unique values:")
for col in BINARY_COLS:
    print(f"  {col}: {sorted(df[col].unique())}")

Post-transformation statistics (continuous + ordinal):
           count     mean     std     min      25%     50%      75%      max   skew
Edad      2376.0   66.114  18.114  18.000   54.000   69.00   81.000   98.000 -0.609
FC        2376.0   89.739  29.998  16.000   70.000   85.00  103.000  267.000  1.194
FR        2376.0   19.628   8.426   3.000   14.000   18.00   23.000   60.000  1.304
FiO2      2376.0    0.304   0.205   0.210    0.210    0.21    0.280    1.000  2.643
Glucemia  2376.0  144.099  60.101  22.000  107.000  128.00  163.000  640.000  2.577
Lactato   2376.0    3.549   2.575   0.900    2.000    2.90    4.100   20.400  2.644
SpO2      2376.0   93.327   8.604  45.000   93.000   96.00   98.000  100.000 -2.925
TAD       2376.0   79.307  18.337  20.000   68.000   80.00   90.000  183.000  0.112
TAM       2376.0   98.854  20.963  27.333   86.333   99.00  111.667  195.333  0.098
TAS       2376.0  137.982  30.640  40.000  119.000  137.00  155.250  261.000  0.282
TT        2376.0   36

### 12.1 Before vs. after: continuous variable distributions

Overlay histograms confirm the normalisation mapped all continuous features to [0, 1] while preserving distributional shape.

In [ ]:
# ── §12.1 — Post-normalisation distribution plots ────────────────────────
# Controlled by CFG_MINMAX and CFG_PLOT
if CFG_MINMAX and CFG_PLOT:
    df_raw = pd.read_csv(DATA_PATH)

    n_cont       = len(CONTINUOUS_COLS)
    n_cols_plot  = 3
    n_rows_plot  = (n_cont + n_cols_plot - 1) // n_cols_plot

    fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                              figsize=(4 * n_cols_plot, 3 * n_rows_plot))
    axes = axes.flatten()

    for i, col in enumerate(CONTINUOUS_COLS):
        ax = axes[i]
        ax.hist(df[col], bins=40, color=TFG_COLORS["after"],
                edgecolor="white", alpha=0.8, label="Transformed")
        ax.set_title(col, fontsize=10)
        ax.set_ylabel("Freq.")
        if i == 0:
            ax.legend(fontsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Post-transformation distributions (continuous columns, [0, 1])",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
    del df_raw
elif not CFG_MINMAX:
    print("— CFG_MINMAX=False: normalisation plot skipped.")
elif not CFG_PLOT:
    print("— CFG_PLOT=False: normalisation plot skipped.")


— CFG_MINMAX=False: normalisation plot skipped.


## **13. Metadata export for generative models**

Both CTAB-GAN+ and ARF require explicit metadata about variable types and transformation parameters. We export:

1. **`column_types`** — Classification of each column as `"continuous"`, `"binary"`, or `"ordinal"`.
2. **`scaler_params`** — MinMaxScaler parameters (`min`, `max`, `scale`) for reversibility.
3. **`log_transforms`** — Which columns were log1p-transformed and their inverse function.
4. **`generation_order`** — Recommended column order for Sequential RF synthesis (demographics → vitals → interventions → outcomes).
5. **`dropped_columns`** — Columns removed and rationale, for documentation.

> *Why export metadata as JSON?* JSON is human-readable, language-agnostic, and can be loaded in other notebook files. Storing the exact scaler parameters avoids numerical discrepancies when inverse-transforming synthetic data back to clinical units.

In [ ]:
"""
PURPOSE : Build the metadata dictionary exported alongside the transformed dataset.

Output  : metadata (dict) — column types, applied-transformation flags, scaler
          parameters, log-transform info, generation order, and OHE column mappings
          needed to inverse-transform synthetic data in NB8–NB10.
"""

# ── §13 — Build metadata dictionary ──────────────────────────────────────
# Reflects only the transformations that were actually applied,
# as determined by the configuration flags in §2.5.

# Column-type classification (always present)
column_types: dict = {}
for col in df.columns:
    if col in CONTINUOUS_COLS:
        column_types[col] = "continuous"
    elif col in ORDINAL_COLS:
        column_types[col] = "ordinal"
    elif col in BINARY_COLS or col in ALL_MORT_COLS:
        column_types[col] = "binary"
    elif col in ritmo_ohe_cols or col in st_ohe_cols:
        column_types[col] = "ohe_binary"
    else:
        column_types[col] = "categorical"

# Generation order
_ohe_all = ritmo_ohe_cols + st_ohe_cols
GENERATION_ORDER = list(
    CONTINUOUS_COLS
    + ORDINAL_COLS
    + BINARY_COLS
    + ([c for c in df.columns if c in CATEGORICAL_COLS] if not CFG_OHE else [])
    + _ohe_all
    + ["Mort. 2D", "Mort. 7D", "Mort. 30D"]
)

assert set(GENERATION_ORDER) == set(df.columns), \
    f"Generation order mismatch: {set(df.columns) - set(GENERATION_ORDER)}"

# Applied-transformation flags (for downstream notebooks)
_applied = {
    "drop_tam":   CFG_DROP_TAM,
    "log1p":      CFG_LOG1P,
    "int_cast":   CFG_INT_CAST,
    "ohe":        CFG_OHE,
    "minmax":     CFG_MINMAX,
}

metadata = {
    "source_file":         "dataset_FINAL.csv",
    "notebook":            "NB5 — Pre-Generation Data Transformations",
    "applied_transforms":  _applied,
    "n_rows":              int(df.shape[0]),
    "n_cols":              int(df.shape[1]),
    "column_types":        column_types,
    "continuous_columns":  CONTINUOUS_COLS,
    "binary_columns":      BINARY_COLS + ALL_MORT_COLS,
    "ordinal_columns":     ORDINAL_COLS,
    "categorical_columns": CATEGORICAL_COLS if not CFG_OHE else [],
    "ohe_columns":         {"Ritmo": ritmo_ohe_cols, "ST": st_ohe_cols},
    "outcome_columns":     ALL_MORT_COLS,
    "scaler_params":       scaler_metadata,
    "log_transforms":      log_metadata,
    "ohe_metadata":        ohe_metadata,
    "dropped_columns":     _dropped_cols,
    "generation_order":    GENERATION_ORDER,
    "mortality_rates":     {col: round(float(df[col].mean()), 4) for col in ALL_MORT_COLS},
}

print("Metadata summary:")
print(f"  Applied transforms: {_applied}")
print(f"  Total columns:      {metadata['n_cols']}")
print(f"  Continuous:         {len(CONTINUOUS_COLS)}")
print(f"  Binary:             {len(BINARY_COLS) + len(ALL_MORT_COLS)}")
print(f"  OHE (Ritmo):        {len(ritmo_ohe_cols)}")
print(f"  OHE (ST):           {len(st_ohe_cols)}")
print(f"  Ordinal:            {len(ORDINAL_COLS)}")
print(f"  Generation order:   {len(GENERATION_ORDER)} columns")


Metadata summary:
  Applied transforms: {'drop_tam': False, 'log1p': False, 'int_cast': True, 'ohe': True, 'minmax': False}
  Total columns:      55
  Continuous:         11
  Binary:             16
  OHE (Ritmo):        18
  OHE (ST):           6
  Ordinal:            4
  Generation order:   55 columns


## **14. Export the dataset and metadata**

Two files are saved:

| File | Description |
|------|-------------|
| `dataset_PREGEN.csv` | Transformed dataset (2 376 × 32) ready for synthetic generation |
| `pregen_metadata.json` | Full transformation metadata for reversibility |

> *Note:* The generation notebooks (**NB6** for CTAB-GAN+)  must load both files. The metadata JSON contains everything needed to inverse-transform synthetic data back to the original clinical scale: scaler parameters, log-transform identifiers, and column type classifications.

In [ ]:
# ── Output paths ──────────────────────────────────────────────────────
OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/TFG/"
CSV_OUT = OUT_DIR + "dataset_PREGEN.csv"
META_OUT = OUT_DIR + "pregen_metadata.json"

# ── Save preprocessed dataset ────────────────────────────────────────
df.to_csv(CSV_OUT, index=False)
print(f"✓ Dataset saved: {CSV_OUT}")
print(f"  Shape: {df.shape}")

# ── Save metadata ────────────────────────────────────────────────────
with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"✓ Metadata saved: {META_OUT}")

✓ Dataset saved: /content/drive/MyDrive/Colab Notebooks/TFG/dataset_PREGEN.csv
  Shape: (2376, 55)
✓ Metadata saved: /content/drive/MyDrive/Colab Notebooks/TFG/pregen_metadata.json


## **15. Final verification**

Reload the exported files and confirm data integrity round-trips correctly.

In [ ]:
# ── §15 — Final verification ─────────────────────────────────────────────
df_check = pd.read_csv(CSV_OUT)
with open(META_OUT, "r") as f:
    meta_check = json.load(f)

# Shape
assert df_check.shape == df.shape, "Shape mismatch after reload"
print(f"✓ Shape match: {df_check.shape}")

# NaN
assert df_check.isna().sum().sum() == 0, "NaN found after reload"
print(f"✓ No NaN after reload")

# Mortality rates preserved
for col in ALL_MORT_COLS:
    original_rate = df[col].mean()
    reloaded_rate = df_check[col].mean()
    assert abs(original_rate - reloaded_rate) < 1e-6, \
        f"Mortality rate mismatch for {col}"
    print(f"✓ {col} rate preserved: {reloaded_rate:.4f}")

# Metadata completeness
required_keys = ["column_types", "scaler_params", "log_transforms",
                 "generation_order", "outcome_columns", "applied_transforms"]
for key in required_keys:
    assert key in meta_check, f"Missing metadata key: {key}"
print(f"✓ All {len(required_keys)} required metadata keys present")

# Continuous range check (only if MinMax was applied)
if meta_check["applied_transforms"].get("minmax", False):
    for col in meta_check["continuous_columns"]:
        assert df_check[col].min() >= -1e-9, f"{col} below 0"
        assert df_check[col].max() <= 1.0 + 1e-9, f"{col} above 1"
    print(f"✓ All continuous columns in [0, 1]")
else:
    print("— MinMax skipped: continuous range check not applicable.")

print("\n" + "=" * 60)
print("NB5 COMPLETE — dataset_PREGEN.csv is ready for generation.")
print("=" * 60)


✓ Shape match: (2376, 55)
✓ No NaN after reload
✓ Mort. 2D rate preserved: 0.0480
✓ Mort. 7D rate preserved: 0.0728
✓ Mort. 30D rate preserved: 0.1098
✓ All 6 required metadata keys present
— MinMax skipped: continuous range check not applicable.

NB5 COMPLETE — dataset_PREGEN.csv is ready for generation.


## **Appendix A — Inverse transformation demo**

Demonstration of how to reverse all transformations applied in this notebook. This exact code will be used in the evaluation notebook after synthetic data generation to recover clinically interpretable values.

The inverse pipeline is:
1. **MinMaxScaler inverse** — `x_original = x_scaled * data_range + data_min`
2. **expm1 inverse** — `x_original = exp(x_log) - 1` for Lactato and Glucemia

In [ ]:
def inverse_transform(df_synth: pd.DataFrame,
                      meta: dict) -> pd.DataFrame:
    """Reverse all NB5 transformations to recover clinical-scale values.

    Applies the inverse of each applied transformation in reverse order.

    Parameters
    ----------
    df_synth : pd.DataFrame
        Synthetic (or real) dataset in the transformed NB5 space.
    meta     : dict
        Metadata dictionary from §13 (loaded from ``pregen_metadata.json``).

    Returns
    -------
    pd.DataFrame
        Dataset with variables restored to their original clinical scale:
        mmol/L for Lactato, mg/dL for Glucemia, integer code for Ritmo / ST.

    Inverse pipeline
    ----------------
    1. MinMaxScaler inverse — ``x = x_scaled * data_range + data_min``
    2. expm1 inverse        — ``x = exp(x_log) - 1`` (Lactato, Glucemia)
    3. OHE inverse          — argmax over k dummy cols → original integer code

    Notes
    -----
    All inverse parameters are stored in ``meta["scaler_params"]`` and
    ``meta["log_transforms"]``, enabling exact numerical round-trip recovery.
    """
    df_inv = df_synth.copy()

    # Step 1: Inverse MinMaxScaler
    for col in meta["continuous_columns"]:
        if col in df_inv.columns and col in meta["scaler_params"]:
            params = meta["scaler_params"][col]
            df_inv[col] = df_inv[col] * params["data_range"] + params["min"]

    # Step 2: Inverse log1p → expm1
    for col, info in meta["log_transforms"].items():
        if col in df_inv.columns and info["inverse"] == "expm1":
            df_inv[col] = np.expm1(df_inv[col])

    # Step 3: Inverse OHE → recover original integer code
    for orig_col, ohe_info in meta.get("ohe_metadata", {}).items():
        col_map = ohe_info["column_map"]  # {int_code: ohe_col_name}
        ohe_cols = [c for c in df_inv.columns if c in col_map.values()]
        if ohe_cols:
            inv_map = {v: int(k) for k, v in col_map.items()}  # keys are str after JSON round-trip  # ohe_col_name → int
            # argmax across OHE cols → index → map back to int code
            ohe_matrix = df_inv[ohe_cols].values
            best_idx = np.argmax(ohe_matrix, axis=1)
            df_inv[orig_col] = [inv_map[ohe_cols[i]] for i in best_idx]
            df_inv = df_inv.drop(columns=ohe_cols)

    return df_inv

# ── Demo: round-trip on the real data ────────────────────────────────
df_recovered = inverse_transform(df.copy(), metadata)

# Compare continuous columns against original
drop_for_compare = [COL_TO_DROP] if CFG_DROP_TAM else []
df_original = pd.read_csv(DATA_PATH).drop(columns=drop_for_compare, errors="ignore")
print("Round-trip verification — continuous columns (max abs error):")
for col in CONTINUOUS_COLS:
    if col in df_original.columns:
        max_err = (df_recovered[col] - df_original[col]).abs().max()
        status = "✓" if max_err < 0.01 else "✗"
        print(f"  {status} {col}: {max_err:.6f}")

# Check OHE inverse recovers original integer codes
print("\nRound-trip verification — categorical columns (exact match %):")
for orig_col in ["Ritmo", "ST"]:
    if orig_col in df_original.columns and orig_col in df_recovered.columns:
        match = (df_recovered[orig_col] == df_original[orig_col]).mean() * 100
        status = "✓" if match == 100.0 else "✗"
        print(f"  {status} {orig_col}: {match:.1f}% exact match")

del df_original, df_recovered

Round-trip verification — continuous columns (max abs error):
  ✓ Edad: 0.000000
  ✓ FC: 0.000000
  ✓ FR: 0.000000
  ✓ FiO2: 0.000000
  ✓ Glucemia: 0.000000
  ✓ Lactato: 0.000000
  ✓ SpO2: 0.000000
  ✓ TAD: 0.000000
  ✓ TAM: 0.000000
  ✓ TAS: 0.000000
  ✓ TT: 0.000000

Round-trip verification — categorical columns (exact match %):
  ✓ Ritmo: 100.0% exact match
  ✓ ST: 100.0% exact match
